# Polynomial Regression으로 Calories_Burned 예측
각 fold에서 Polynomial과 Scaling을 train fold로만 학습한 뒤, 

원단위 Ridge와 sqrt-Ridge 두 모델의 OOF 예측을 생성하고, 

최적 가중치로 결합하여 최종 예측

최종 RMSE : 0.29

## 1. 라이브러리 임포트, 랜덤 시드 , 상수 고정

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from scipy.optimize import minimize

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

RANDOM_STATE = 42
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

## 2. 데이터 로드

In [2]:
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")
submission = pd.read_csv("sample_submission.csv")

## 3. Feature Engineering Function - 수정금지

In [3]:
def create_features_safe(df):
    df = df.copy()

    df['Height_Total_Inches'] = df['Height(Feet)'] * 12 + df['Height(Remainder_Inches)']
    df['Temp_diff'] = df['Body_Temperature(F)'] - 98.6

    df['Duration_bin'] = pd.cut(
        df['Exercise_Duration'],
        bins=[-np.inf, 10, 20, 30, np.inf],
        labels=[0, 1, 2, 3]
    ).astype(int)

    df['Duration_x_BPM'] = df['Exercise_Duration'] * df['BPM']
    df['Duration_x_TempDiff'] = df['Exercise_Duration'] * df['Temp_diff']
    df['BPM_x_TempDiff'] = df['BPM'] * df['Temp_diff']

    # 최종 TOP5에 쓰는 핵심 변수명 고정
    df["Intensity"] = df["Duration_x_BPM"]
    df["Effort"] = df["Weight(lb)"] * df["Intensity"]

    df['Duration_sq'] = df['Exercise_Duration'] ** 2
    df['Temp_diff_sq'] = df['Temp_diff'] ** 2
    df['Dur_BPM_TempDiff'] = df['Exercise_Duration'] * df['BPM'] * df['Temp_diff']

    df['BPM_per_Duration'] = df['BPM'] / (df['Exercise_Duration'] + 1)
    df['TempDiff_per_Duration'] = df['Temp_diff'] / (df['Exercise_Duration'] + 1)

    h2 = (df['Height_Total_Inches'] ** 2).replace(0, np.nan)
    df['BMI'] = 703 * df['Weight(lb)'] / h2
    df['BMI'] = df['BMI'].fillna(df['BMI'].median())

    df['Weight_x_Duration'] = df['Weight(lb)'] * df['Exercise_Duration']

    df['Log_BPM'] = np.log1p(df['BPM'])
    df['Log_Duration'] = np.log1p(df['Exercise_Duration'])
    df['Log_Weight_BPM_Dur'] = np.log1p(df['Weight(lb)'] * df['BPM'] * df['Exercise_Duration'])

    # # 저칼로리 구간: 짧은 Duration 정밀도 향상
    # df['Duration_sqrt'] = np.sqrt(df['Exercise_Duration'])   # 1분↔2분↔3분 차이 증폭
    # df['Duration_cbrt'] = np.cbrt(df['Exercise_Duration'])   # 더 극단적 짧은 운동 강조

    # # 고칼로리 구간: 고체중 + 저연령 조합 포착
    # df['Weight_div_Age'] = df['Weight(lb)'] / (df['Age'] + 1)
    # df['Effort_div_Age'] = df['Effort'] / (df['Age'] + 1)   
    return df

타겟이 정수라는 점에 착안해, round-only를 적용한 뒤 라운딩 경계(0.5±0.004)에서만 보조 예측(sqrt base)으로 tie-break하는 규칙 기반 보정을 추가했고 OOF RMSE를 0.218까지 개선

In [24]:
# Duration × BPM × Temp 조합으로 칼로리가 설명되는지 확인
train['DBT'] = train['Exercise_Duration'] * train['BPM'] * train['Body_Temperature(F)']
train['DB']  = train['Exercise_Duration'] * train['BPM']
train['DT']  = train['Exercise_Duration'] * train['Body_Temperature(F)']

print("Duration*BPM*Temp 상관:", train['DBT'].corr(train['Calories_Burned']))
print("Duration*BPM 상관:     ", train['DB'].corr(train['Calories_Burned']))
print("Duration*Temp 상관:    ", train['DT'].corr(train['Calories_Burned']))

# 단순 선형회귀로 DBT 하나만으로 얼마나 설명되는지
from numpy.polynomial import polynomial as P
x = train['DBT'].values
y = train['Calories_Burned'].values
coeffs = np.polyfit(x, y, 1)
pred = np.polyval(coeffs, x)
print("\nDBT 단일 변수 선형 RMSE:", rmse(y, pred))
print("계수:", coeffs)

Duration*BPM*Temp 상관: 0.9745517273707681
Duration*BPM 상관:      0.9743775693320909
Duration*Temp 상관:     0.9553693226183922

DBT 단일 변수 선형 RMSE: 14.080310799068853
계수: [ 6.20397564e-04 -1.12997241e+01]


In [25]:
# Temp를 분리해서 확인
train['Temp_diff'] = train['Body_Temperature(F)'] - 98.6  # 정상체온 편차

# Temp_diff가 잔차를 설명하는지 확인
base_pred = train['Exercise_Duration'] * train['BPM'] * 6.20397564e-04 - 11.2997241
residual = train['Calories_Burned'] - base_pred

print("잔차 vs Temp_diff 상관:", residual.corr(train['Temp_diff']))
print("잔차 vs Age 상관:      ", residual.corr(train['Age']))
print("잔차 vs Weight 상관:   ", residual.corr(train['Weight(lb)']))
print("잔차 vs Gender 상관:   ", residual.corr((train['Gender'] == train['Gender'].unique()[0]).astype(int)))

print("\n잔차 통계:")
print(residual.describe())
print("\n잔차 절댓값 평균:", residual.abs().mean())

잔차 vs Temp_diff 상관: 0.8236818731710932
잔차 vs Age 상관:       0.16089665706654258
잔차 vs Weight 상관:    0.042914055894792795
잔차 vs Gender 상관:    -0.02778454116986571

잔차 통계:
count    7500.000000
mean       99.712384
std        62.254037
min        12.214109
25%        45.825120
50%        87.464049
75%       147.944311
max       309.122749
dtype: float64

잔차 절댓값 평균: 99.71238395448849


잔차 평균이 99.7이고 Temp_diff 상관이 0.82. Temp_diff가 잔차의 핵심

In [26]:
# 잔차를 Temp_diff로 회귀
x2 = train['Temp_diff'].values
coeffs2 = np.polyfit(x2, residual.values, 1)
print("Temp_diff 계수:", coeffs2)

# 2단계 예측
pred2 = base_pred + np.polyval(coeffs2, x2)
print("2변수 RMSE (DB + Temp_diff):", rmse(train['Calories_Burned'].values, pred2))

# 잔차2 분석
residual2 = train['Calories_Burned'] - pred2
print("\n잔차2 vs Temp_diff 상관:", residual2.corr(train['Temp_diff']))
print("잔차2 vs Duration 상관:  ", residual2.corr(train['Exercise_Duration']))
print("잔차2 vs BPM 상관:       ", residual2.corr(train['BPM']))
print("잔차2 vs Age 상관:       ", residual2.corr(train['Age']))
print("잔차2 vs Weight 상관:    ", residual2.corr(train['Weight(lb)']))
print("\n잔차2 통계:")
print(residual2.describe())

Temp_diff 계수: [ 36.29380549 -97.4926697 ]
2변수 RMSE (DB + Temp_diff): 35.29893348928819

잔차2 vs Temp_diff 상관: 1.4434022051999524e-14
잔차2 vs Duration 상관:   0.3691536159635518
잔차2 vs BPM 상관:        0.46722753965161196
잔차2 vs Age 상관:        0.25981304184634674
잔차2 vs Weight 상관:     0.06470655233197563

잔차2 통계:
count    7.500000e+03
mean     9.121550e-13
std      3.530129e+01
min     -8.915780e+01
25%     -2.531181e+01
50%     -6.784035e+00
75%      2.120974e+01
max      1.476419e+02
dtype: float64


Temp_diff를 선형으로 더하면 안 되는 구조였나
Temp가 곱해지는 형태일 가능성이 높다

In [27]:
# Temp_diff가 곱셈으로 들어가는지 확인
# 가설: Calories = a * Duration * BPM * f(Temp) + b

# 방법1: Duration*BPM*Temp_diff
train['DB_Tdiff'] = train['Exercise_Duration'] * train['BPM'] * train['Temp_diff']
print("Duration*BPM*Temp_diff 상관:", train['DB_Tdiff'].corr(train['Calories_Burned']))

# 방법2: (Duration*BPM) + (Duration*BPM*Temp_diff) 조합
X = np.column_stack([
    train['Exercise_Duration'] * train['BPM'],
    train['Exercise_Duration'] * train['BPM'] * train['Temp_diff'],
])
from numpy.linalg import lstsq
coeffs3, _, _, _ = lstsq(np.column_stack([X, np.ones(len(X))]), train['Calories_Burned'].values, rcond=None)
pred3 = X @ coeffs3[:2] + coeffs3[2]
print("DB + DB*Tdiff 조합 RMSE:", rmse(train['Calories_Burned'].values, pred3))
print("계수:", coeffs3)

# 잔차3
residual3 = train['Calories_Burned'] - pred3
print("\n잔차3 vs Duration 상관:", residual3.corr(train['Exercise_Duration']))
print("잔차3 vs BPM 상관:      ", residual3.corr(train['BPM']))
print("잔차3 vs Age 상관:      ", residual3.corr(train['Age']))
print("잔차3 vs Weight 상관:   ", residual3.corr(train['Weight(lb)']))
print("잔차3 통계:\n", residual3.describe())

Duration*BPM*Temp_diff 상관: 0.9708901966205096
DB + DB*Tdiff 조합 RMSE: 14.035111042084809
계수: [ 5.06886864e-02  2.05268122e-03 -8.77346210e+00]

잔차3 vs Duration 상관: -0.05568936674465559
잔차3 vs BPM 상관:       0.1493426677868983
잔차3 vs Age 상관:       0.6339130642517627
잔차3 vs Weight 상관:    0.17148798933940415
잔차3 통계:
 count    7.500000e+03
mean     4.292815e-14
std      1.403605e+01
min     -5.354244e+01
25%     -7.859393e+00
50%      1.214055e-01
75%      6.186209e+00
max      8.420455e+01
Name: Calories_Burned, dtype: float64


In [28]:
# Age가 곱셈으로 들어가는지 확인
X2 = np.column_stack([
    train['Exercise_Duration'] * train['BPM'],
    train['Exercise_Duration'] * train['BPM'] * train['Temp_diff'],
    train['Exercise_Duration'] * train['BPM'] * train['Age'],
    train['Exercise_Duration'] * train['BPM'] * train['Temp_diff'] * train['Age'],
    train['Age'],
])
coeffs4, _, _, _ = lstsq(np.column_stack([X2, np.ones(len(X2))]), train['Calories_Burned'].values, rcond=None)
pred4 = X2 @ coeffs4[:5] + coeffs4[5]
print("+ Age 조합 RMSE:", rmse(train['Calories_Burned'].values, pred4))
print("계수:", coeffs4)

residual4 = train['Calories_Burned'] - pred4
print("\n잔차4 vs Duration 상관:", residual4.corr(train['Exercise_Duration']))
print("잔차4 vs BPM 상관:      ", residual4.corr(train['BPM']))
print("잔차4 vs Age 상관:      ", residual4.corr(train['Age']))
print("잔차4 vs Weight 상관:   ", residual4.corr(train['Weight(lb)']))
print("잔차4 vs Temp_diff 상관:", residual4.corr(train['Temp_diff']))
print("잔차4 통계:\n", residual4.describe())

+ Age 조합 RMSE: 9.75587330888903
계수: [ 4.02591830e-02  1.68109022e-03  2.59117006e-04  6.01730057e-06
  6.47166011e-02 -1.14770502e+01]

잔차4 vs Duration 상관: -0.07888336094739422
잔차4 vs BPM 상관:       0.2141073261363517
잔차4 vs Age 상관:       7.586519263718638e-16
잔차4 vs Weight 상관:    0.16866464921863455
잔차4 vs Temp_diff 상관: -0.1956186160078668
잔차4 통계:
 count    7.500000e+03
mean    -8.041828e-14
std      9.756524e+00
min     -3.727473e+01
25%     -5.780295e+00
50%      5.145790e-01
75%      5.401117e+00
max      6.160701e+01
Name: Calories_Burned, dtype: float64


In [29]:
# Weight와 BPM*Temp_diff 추가
X3 = np.column_stack([
    train['Exercise_Duration'] * train['BPM'],
    train['Exercise_Duration'] * train['BPM'] * train['Temp_diff'],
    train['Exercise_Duration'] * train['BPM'] * train['Age'],
    train['Exercise_Duration'] * train['BPM'] * train['Temp_diff'] * train['Age'],
    train['Age'],
    train['Exercise_Duration'] * train['Weight(lb)'],
    train['Exercise_Duration'] * train['BPM'] * train['Weight(lb)'],
    train['Weight(lb)'],
    train['BPM'] * train['Temp_diff'],
    train['BPM'],
])
coeffs5, _, _, _ = lstsq(np.column_stack([X3, np.ones(len(X3))]), train['Calories_Burned'].values, rcond=None)
pred5 = X3 @ coeffs5[:10] + coeffs5[10]
print("RMSE:", rmse(train['Calories_Burned'].values, pred5))

residual5 = train['Calories_Burned'] - pred5
print("\n잔차5 vs Duration 상관:", residual5.corr(train['Exercise_Duration']))
print("잔차5 vs BPM 상관:      ", residual5.corr(train['BPM']))
print("잔차5 vs Age 상관:      ", residual5.corr(train['Age']))
print("잔차5 vs Weight 상관:   ", residual5.corr(train['Weight(lb)']))
print("잔차5 vs Temp_diff 상관:", residual5.corr(train['Temp_diff']))
print("잔차5 통계:\n", residual5.describe())

RMSE: 6.081502918651104

잔차5 vs Duration 상관: 0.0010066916868624164
잔차5 vs BPM 상관:       6.5512878755610296e-15
잔차5 vs Age 상관:       1.9573754331910574e-14
잔차5 vs Weight 상관:    3.0702346931416254e-14
잔차5 vs Temp_diff 상관: 6.416242710818286e-05
잔차5 통계:
 count    7.500000e+03
mean     1.832935e-13
std      6.081908e+00
min     -3.849792e+01
25%     -3.163053e+00
50%      8.480704e-02
75%      3.173558e+00
max      2.684464e+01
Name: Calories_Burned, dtype: float64


BPM, Age, Weight, Temp_diff 잔차 상관이 전부 0에 수렴했다! Duration만 살짝 남음 (0.001). 

In [30]:
# Duration 비선형 항 추가
X4 = np.column_stack([
    train['Exercise_Duration'] * train['BPM'],
    train['Exercise_Duration'] * train['BPM'] * train['Temp_diff'],
    train['Exercise_Duration'] * train['BPM'] * train['Age'],
    train['Exercise_Duration'] * train['BPM'] * train['Temp_diff'] * train['Age'],
    train['Age'],
    train['Exercise_Duration'] * train['Weight(lb)'],
    train['Exercise_Duration'] * train['BPM'] * train['Weight(lb)'],
    train['Weight(lb)'],
    train['BPM'] * train['Temp_diff'],
    train['BPM'],
    train['Exercise_Duration'] ** 2,
    train['Exercise_Duration'] ** 2 * train['BPM'],
    train['Exercise_Duration'],
])
coeffs6, _, _, _ = lstsq(np.column_stack([X4, np.ones(len(X4))]), train['Calories_Burned'].values, rcond=None)
pred6 = X4 @ coeffs6[:13] + coeffs6[13]
print("RMSE:", rmse(train['Calories_Burned'].values, pred6))

residual6 = train['Calories_Burned'] - pred6
print("\n잔차6 vs Duration 상관:  ", residual6.corr(train['Exercise_Duration']))
print("잔차6 vs Duration^2 상관:", residual6.corr(train['Exercise_Duration']**2))
print("잔차6 vs BPM 상관:       ", residual6.corr(train['BPM']))
print("잔차6 vs Weight 상관:    ", residual6.corr(train['Weight(lb)']))
print("잔차6 vs Temp_diff 상관: ", residual6.corr(train['Temp_diff']))
print("잔차6 통계:\n", residual6.describe())

RMSE: 6.074083005068506

잔차6 vs Duration 상관:   3.655474645277629e-14
잔차6 vs Duration^2 상관: 4.2057728103987466e-14
잔차6 vs BPM 상관:        3.355494687768562e-14
잔차6 vs Weight 상관:     2.3016712930406553e-14
잔차6 vs Temp_diff 상관:  -0.00012916017581672787
잔차6 통계:
 count    7.500000e+03
mean     1.746230e-14
std      6.074488e+00
min     -3.830304e+01
25%     -3.171438e+00
50%      4.592492e-02
75%      3.185222e+00
max      2.873437e+01
Name: Calories_Burned, dtype: float64


모든 상관관계가 0에 수렴했는데 잔차 std가 6으로 남아있다는 건 설명 불가능한 노이즈가 데이터에 내재되어 있다는 뜻

In [31]:
# 잔차6의 분포 확인 - 노이즈인지 아니면 빠진 변수가 있는지
import matplotlib.pyplot as plt

print("잔차6 히스토그램 분포:")
print(pd.cut(residual6, bins=20).value_counts().sort_index())

# 혹시 Gender가 영향을 주는지
male_mask2 = train['Gender'] == train['Gender'].unique()[0]
print("\n성별 unique:", train['Gender'].unique())
print("잔차6 남성 std:", residual6[male_mask2].std())
print("잔차6 여성 std:", residual6[~male_mask2].std())
print("잔차6 남성 mean:", residual6[male_mask2].mean())
print("잔차6 여성 mean:", residual6[~male_mask2].mean())

# Weight_Status도 확인
print("\nWeight_Status unique:", train['Weight_Status'].unique())
for ws in train['Weight_Status'].unique():
    mask_ws = train['Weight_Status'] == ws
    print(f"{ws} 잔차 mean/std: {residual6[mask_ws].mean():.3f} / {residual6[mask_ws].std():.3f}")

잔차6 히스토그램 분포:
Calories_Burned
(-38.37, -34.951]        6
(-34.951, -31.599]       1
(-31.599, -28.247]       4
(-28.247, -24.896]       8
(-24.896, -21.544]      16
(-21.544, -18.192]      30
(-18.192, -14.84]       53
(-14.84, -11.488]      135
(-11.488, -8.136]      307
(-8.136, -4.784]       686
(-4.784, -1.432]      1499
(-1.432, 1.919]       2204
(1.919, 5.271]        1429
(5.271, 8.623]         600
(8.623, 11.975]        296
(11.975, 15.327]       151
(15.327, 18.679]        54
(18.679, 22.031]        15
(22.031, 25.383]         4
(25.383, 28.734]         2
Name: count, dtype: int64

성별 unique: ['F' 'M']
잔차6 남성 std: 6.314552193713933
잔차6 여성 std: 5.820856563341453
잔차6 남성 mean: 0.10515646752715371
잔차6 여성 mean: -0.10622747952551903

Weight_Status unique: ['Normal Weight' 'Overweight' 'Obese']
Normal Weight 잔차 mean/std: -0.343 / 5.993
Overweight 잔차 mean/std: 0.668 / 6.211
Obese 잔차 mean/std: -0.690 / 5.418


잔차가 정규분포 모양이고, 성별/Weight_Status별 mean이 거의 0, std도 고르게 6 내외

In [32]:
# train/test ID 범위 확인 - 혹시 test가 train과 겹치는지
print("train ID range:", train['ID'].min(), "~", train['ID'].max())
print("test  ID range:", test['ID'].min(), "~", test['ID'].max())
print("train 행수:", len(train))
print("test  행수:", len(test))

# train에서 동일한 피처 조합이 중복되는지
feature_cols = ['Age', 'Exercise_Duration', 'BPM', 'Weight(lb)', 'Body_Temperature(F)', 'Gender']
dup = train.duplicated(subset=feature_cols, keep=False)
print("\n피처 완전 동일한 중복 행수:", dup.sum())

# test 행이 train에 존재하는지
merged = test[feature_cols].merge(train[feature_cols + ['Calories_Burned']], on=feature_cols, how='left')
print("test 중 train에 exact match 있는 행수:", merged['Calories_Burned'].notna().sum())
print("test 전체 행수:", len(test))

train ID range: TRAIN_0000 ~ TRAIN_7499
test  ID range: TEST_0000 ~ TEST_7499
train 행수: 7500
test  행수: 7500

피처 완전 동일한 중복 행수: 6
test 중 train에 exact match 있는 행수: 6
test 전체 행수: 7500


In [33]:
# 혹시 원본 데이터셋(Kaggle)이 있는지 확인
# 이 대회 데이터가 Kaggle의 "Calories Burned" 데이터셋을 기반으로 했을 가능성

# train 피처 조합으로 칼로리를 거의 완벽히 예측하는 외부 데이터가 있다면
# test의 정답을 사실상 알 수 있음

# 우선 train 자체에서 동일 피처 → 칼로리가 항상 같은지 확인
print("=== 피처 동일한데 칼로리 다른 경우 ===")
dupes = train[train.duplicated(subset=feature_cols, keep=False)].sort_values(feature_cols)
print(dupes[feature_cols + ['Calories_Burned']])

# 그리고 칼로리 분포 확인 - 정수인지 소수인지
print("\n칼로리 샘플 (소수점 확인):")
print(train['Calories_Burned'].head(20).tolist())
print("\n칼로리가 정수인 비율:", (train['Calories_Burned'] == train['Calories_Burned'].round()).mean())

=== 피처 동일한데 칼로리 다른 경우 ===
      Age  Exercise_Duration    BPM  Weight(lb)  Body_Temperature(F) Gender  \
3433   20               28.0  102.0       134.5                105.8      F   
3894   20               28.0  102.0       134.5                105.8      F   
1793   25                6.0   89.0       143.3                102.6      F   
2358   25                6.0   89.0       143.3                102.6      F   
4251   28                9.0   90.0       130.1                103.1      F   
4662   28                9.0   90.0       130.1                103.1      F   

      Calories_Burned  
3433            155.0  
3894            155.0  
1793             25.0  
2358             25.0  
4251             40.0  
4662             40.0  

칼로리 샘플 (소수점 확인):
[166.0, 33.0, 23.0, 91.0, 32.0, 71.0, 18.0, 81.0, 172.0, 29.0, 60.0, 44.0, 107.0, 8.0, 55.0, 144.0, 20.0, 42.0, 79.0, 87.0]

칼로리가 정수인 비율: 1.0


In [34]:
# 잔차6이 어떤 정수 패턴인지 확인
print("잔차6 정수 반올림 후 분포:")
res_round = residual6.round()
print(pd.Series(res_round).value_counts().sort_index().head(20))

# 잔차6과 각 변수의 비선형 관계 탐색
print("\n잔차6 vs Height 관련 변수:")
if 'Height(Feet)' in train.columns:
    print("Height(Feet) 상관:", residual6.corr(train['Height(Feet)']))
    height_total = train['Height(Feet)'] * 12 + train['Height(Remainder_Inches)']
    print("Height_Total_Inches 상관:", residual6.corr(height_total))
    print("BMI 상관:", residual6.corr(703 * train['Weight(lb)'] / (height_total**2)))

# Gender별로 공식이 다른지 확인  
male_mask2 = train['Gender'] == 'M'
print("\n=== 남성만 따로 공식 찾기 ===")
X_male = np.column_stack([
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Age'],
    train.loc[male_mask2, 'Age'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'Weight(lb)'],
    train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'BPM'],
])
c_male, _, _, _ = lstsq(np.column_stack([X_male, np.ones(len(X_male))]), 
                         train.loc[male_mask2, 'Calories_Burned'].values, rcond=None)
pred_male = X_male @ c_male[:7] + c_male[7]
print("남성 RMSE:", rmse(train.loc[male_mask2, 'Calories_Burned'].values, pred_male))

female_mask2 = ~male_mask2
X_female = np.column_stack([
    train.loc[female_mask2, 'Exercise_Duration'] * train.loc[female_mask2, 'BPM'],
    train.loc[female_mask2, 'Exercise_Duration'] * train.loc[female_mask2, 'BPM'] * train.loc[female_mask2, 'Temp_diff'],
    train.loc[female_mask2, 'Exercise_Duration'] * train.loc[female_mask2, 'BPM'] * train.loc[female_mask2, 'Age'],
    train.loc[female_mask2, 'Age'],
    train.loc[female_mask2, 'Exercise_Duration'] * train.loc[female_mask2, 'Weight(lb)'],
    train.loc[female_mask2, 'BPM'] * train.loc[female_mask2, 'Temp_diff'],
    train.loc[female_mask2, 'BPM'],
])
c_female, _, _, _ = lstsq(np.column_stack([X_female, np.ones(len(X_female))]), 
                           train.loc[female_mask2, 'Calories_Burned'].values, rcond=None)
pred_female = X_female @ c_female[:7] + c_female[7]
print("여성 RMSE:", rmse(train.loc[female_mask2, 'Calories_Burned'].values, pred_female))

잔차6 정수 반올림 후 분포:
Calories_Burned
-38.0     2
-37.0     1
-36.0     1
-35.0     3
-31.0     1
-29.0     3
-28.0     1
-27.0     5
-26.0     2
-25.0     2
-24.0     5
-23.0     4
-22.0     6
-21.0     7
-20.0     7
-19.0    10
-18.0    14
-17.0    14
-16.0    14
-15.0    24
Name: count, dtype: int64

잔차6 vs Height 관련 변수:
Height(Feet) 상관: 0.021154687842761966
Height_Total_Inches 상관: -0.033430379820169165
BMI 상관: -0.015936450225917373

=== 남성만 따로 공식 찾기 ===
남성 RMSE: 5.431426193201321
여성 RMSE: 2.022794741288937


In [35]:
# 남성 잔차 분석
res_male = train.loc[male_mask2, 'Calories_Burned'].values - pred_male
res_male_s = pd.Series(res_male, index=train[male_mask2].index)

print("남성 잔차 vs 각 변수 상관:")
for col in ['Age', 'Exercise_Duration', 'BPM', 'Weight(lb)', 'Body_Temperature(F)', 'Temp_diff']:
    print(f"  {col}: {res_male_s.corr(train.loc[male_mask2, col]):.4f}")

height_total = train['Height(Feet)'] * 12 + train['Height(Remainder_Inches)']
print(f"  Height_Total: {res_male_s.corr(height_total[male_mask2]):.4f}")
print(f"  BMI: {res_male_s.corr((703*train['Weight(lb)']/(height_total**2))[male_mask2]):.4f}")

# 남성만 추가 항 탐색
X_male2 = np.column_stack([
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Age'],
    train.loc[male_mask2, 'Age'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'Weight(lb)'],
    train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'BPM'],
    # 추가 항
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Weight(lb)'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Temp_diff'] * train.loc[male_mask2, 'Age'],
    train.loc[male_mask2, 'Weight(lb)'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Age'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Weight(lb)'],
])
c_male2, _, _, _ = lstsq(np.column_stack([X_male2, np.ones(len(X_male2))]),
                          train.loc[male_mask2, 'Calories_Burned'].values, rcond=None)
pred_male2 = X_male2 @ c_male2[:12] + c_male2[12]
print("\n남성 추가항 RMSE:", rmse(train.loc[male_mask2, 'Calories_Burned'].values, pred_male2))

res_male2 = pd.Series(train.loc[male_mask2, 'Calories_Burned'].values - pred_male2, index=train[male_mask2].index)
print("남성 잔차2 상관:")
for col in ['Age', 'Exercise_Duration', 'BPM', 'Weight(lb)', 'Temp_diff']:
    print(f"  {col}: {res_male2.corr(train.loc[male_mask2, col]):.4f}")

남성 잔차 vs 각 변수 상관:
  Age: 0.0000
  Exercise_Duration: -0.0573
  BPM: 0.0000
  Weight(lb): 0.1973
  Body_Temperature(F): -0.0396
  Temp_diff: -0.0396
  Height_Total: 0.1812
  BMI: 0.0780

남성 추가항 RMSE: 1.4544461813973537
남성 잔차2 상관:
  Age: 0.0000
  Exercise_Duration: -0.0110
  BPM: -0.0000
  Weight(lb): -0.0000
  Temp_diff: -0.0154


In [36]:
# Height 추가
X_male3 = np.column_stack([
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Age'],
    train.loc[male_mask2, 'Age'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'Weight(lb)'],
    train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'BPM'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Weight(lb)'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Temp_diff'] * train.loc[male_mask2, 'Age'],
    train.loc[male_mask2, 'Weight(lb)'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Age'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Weight(lb)'],
    # Height 추가
    height_total[male_mask2],
    height_total[male_mask2] * train.loc[male_mask2, 'Exercise_Duration'],
    height_total[male_mask2] * train.loc[male_mask2, 'BPM'],
    height_total[male_mask2] * train.loc[male_mask2, 'Weight(lb)'],
])
c_male3, _, _, _ = lstsq(np.column_stack([X_male3, np.ones(len(X_male3))]),
                          train.loc[male_mask2, 'Calories_Burned'].values, rcond=None)
pred_male3 = X_male3 @ c_male3[:16] + c_male3[16]
print("남성 Height 추가 RMSE:", rmse(train.loc[male_mask2, 'Calories_Burned'].values, pred_male3))

res_male3 = pd.Series(train.loc[male_mask2, 'Calories_Burned'].values - pred_male3, index=train[male_mask2].index)
print("남성 잔차3 상관:")
for col in ['Age', 'Exercise_Duration', 'BPM', 'Weight(lb)', 'Temp_diff']:
    print(f"  {col}: {res_male3.corr(train.loc[male_mask2, col]):.4f}")
print(f"  Height_Total: {res_male3.corr(height_total[male_mask2]):.4f}")
print(f"  BMI: {res_male3.corr((703*train['Weight(lb)']/(height_total**2))[male_mask2]):.4f}")
print("\n잔차3 통계:\n", res_male3.describe())

남성 Height 추가 RMSE: 1.3058999275343088
남성 잔차3 상관:
  Age: -0.0000
  Exercise_Duration: -0.0091
  BPM: 0.0000
  Weight(lb): 0.0000
  Temp_diff: -0.0149
  Height_Total: 0.0000
  BMI: -0.0011

잔차3 통계:
 count    3.731000e+03
mean    -1.700503e-13
std      1.306075e+00
min     -1.003178e+01
25%     -6.626553e-01
50%     -1.446812e-02
75%      6.279920e-01
max      8.645817e+00
dtype: float64


In [37]:
# 여성도 Height 추가해서 개선
X_female2 = np.column_stack([
    train.loc[female_mask2, 'Exercise_Duration'] * train.loc[female_mask2, 'BPM'],
    train.loc[female_mask2, 'Exercise_Duration'] * train.loc[female_mask2, 'BPM'] * train.loc[female_mask2, 'Temp_diff'],
    train.loc[female_mask2, 'Exercise_Duration'] * train.loc[female_mask2, 'BPM'] * train.loc[female_mask2, 'Age'],
    train.loc[female_mask2, 'Age'],
    train.loc[female_mask2, 'Exercise_Duration'] * train.loc[female_mask2, 'Weight(lb)'],
    train.loc[female_mask2, 'BPM'] * train.loc[female_mask2, 'Temp_diff'],
    train.loc[female_mask2, 'BPM'],
    train.loc[female_mask2, 'Exercise_Duration'] * train.loc[female_mask2, 'BPM'] * train.loc[female_mask2, 'Weight(lb)'],
    train.loc[female_mask2, 'Exercise_Duration'] * train.loc[female_mask2, 'BPM'] * train.loc[female_mask2, 'Temp_diff'] * train.loc[female_mask2, 'Age'],
    train.loc[female_mask2, 'Weight(lb)'] * train.loc[female_mask2, 'Temp_diff'],
    train.loc[female_mask2, 'Age'] * train.loc[female_mask2, 'Temp_diff'],
    train.loc[female_mask2, 'Weight(lb)'],
    height_total[female_mask2],
    height_total[female_mask2] * train.loc[female_mask2, 'Exercise_Duration'],
    height_total[female_mask2] * train.loc[female_mask2, 'BPM'],
    height_total[female_mask2] * train.loc[female_mask2, 'Weight(lb)'],
])
c_female2, _, _, _ = lstsq(np.column_stack([X_female2, np.ones(len(X_female2))]),
                             train.loc[female_mask2, 'Calories_Burned'].values, rcond=None)
pred_female2 = X_female2 @ c_female2[:16] + c_female2[16]
print("여성 RMSE:", rmse(train.loc[female_mask2, 'Calories_Burned'].values, pred_female2))

# 전체 합치기
pred_combined = np.zeros(len(train))
pred_combined[male_mask2.values] = pred_male3
pred_combined[female_mask2.values] = pred_female2
print("전체 RMSE (float):", rmse(train['Calories_Burned'].values, pred_combined))
print("전체 RMSE (round):", rmse(train['Calories_Burned'].values, np.rint(np.clip(pred_combined, 0, None))))

# test 예측 생성
height_total_test = test['Height(Feet)'] * 12 + test['Height(Remainder_Inches)']
test['Temp_diff'] = test['Body_Temperature(F)'] - 98.6
male_test = test['Gender'] == 'M'
female_test = ~male_test

def make_X(df, mask, ht):
    return np.column_stack([
        df.loc[mask, 'Exercise_Duration'] * df.loc[mask, 'BPM'],
        df.loc[mask, 'Exercise_Duration'] * df.loc[mask, 'BPM'] * df.loc[mask, 'Temp_diff'],
        df.loc[mask, 'Exercise_Duration'] * df.loc[mask, 'BPM'] * df.loc[mask, 'Age'],
        df.loc[mask, 'Age'],
        df.loc[mask, 'Exercise_Duration'] * df.loc[mask, 'Weight(lb)'],
        df.loc[mask, 'BPM'] * df.loc[mask, 'Temp_diff'],
        df.loc[mask, 'BPM'],
        df.loc[mask, 'Exercise_Duration'] * df.loc[mask, 'BPM'] * df.loc[mask, 'Weight(lb)'],
        df.loc[mask, 'Exercise_Duration'] * df.loc[mask, 'BPM'] * df.loc[mask, 'Temp_diff'] * df.loc[mask, 'Age'],
        df.loc[mask, 'Weight(lb)'] * df.loc[mask, 'Temp_diff'],
        df.loc[mask, 'Age'] * df.loc[mask, 'Temp_diff'],
        df.loc[mask, 'Weight(lb)'],
        ht[mask],
        ht[mask] * df.loc[mask, 'Exercise_Duration'],
        ht[mask] * df.loc[mask, 'BPM'],
        ht[mask] * df.loc[mask, 'Weight(lb)'],
    ])

X_test_male   = make_X(test, male_test,   height_total_test)
X_test_female = make_X(test, female_test, height_total_test)

test_pred = np.zeros(len(test))
test_pred[male_test.values]   = X_test_male   @ c_male3[:16]   + c_male3[16]
test_pred[female_test.values] = X_test_female @ c_female2[:16] + c_female2[16]

test_pred_round = np.rint(np.clip(test_pred, 0, None))

sub = pd.read_csv("sample_submission.csv")
sub['Calories_Burned'] = test_pred_round.astype(int)
sub.to_csv("submit_formula.csv", index=False)
print("\n저장 완료: submit_formula.csv")
print("test 예측 mean/min/max:", test_pred_round.mean(), test_pred_round.min(), test_pred_round.max())

여성 RMSE: 0.5925602026365854
전체 RMSE (float): 1.0123341089316225
전체 RMSE (round): 1.044796630928718

저장 완료: submit_formula.csv
test 예측 mean/min/max: 89.6628 0.0 325.0


In [38]:
# 남성 잔차3의 패턴 더 깊이 탐색
res_male3_vals = train.loc[male_mask2, 'Calories_Burned'].values - pred_male3

print("남성 잔차3 분포 (정수 반올림):")
print(pd.Series(np.rint(res_male3_vals)).value_counts().sort_index())

# 잔차가 큰 샘플들 확인
male_train = train[male_mask2].copy()
male_train['residual'] = res_male3_vals
male_train['pred'] = pred_male3

print("\n잔차 절댓값 TOP 20:")
print(male_train.nlargest(20, 'residual')[['Age','Exercise_Duration','BPM','Weight(lb)','Body_Temperature(F)','Calories_Burned','pred','residual']])

# 혹시 Duration이 특정 구간에서만 잔차가 큰지
print("\nDuration 구간별 잔차 std:")
male_train['dur_bin'] = pd.cut(male_train['Exercise_Duration'], bins=[0,5,10,15,20,25,30,35])
print(male_train.groupby('dur_bin')['residual'].agg(['mean','std','count']))

남성 잔차3 분포 (정수 반올림):
-10.0       1
-7.0        1
-6.0        4
-5.0        5
-4.0       27
-3.0       80
-2.0      226
-1.0      804
 0.0     1487
 1.0      744
 2.0      231
 3.0       72
 4.0       26
 5.0       16
 6.0        4
 7.0        1
 8.0        1
 9.0        1
Name: count, dtype: int64

잔차 절댓값 TOP 20:
      Age  Exercise_Duration    BPM  Weight(lb)  Body_Temperature(F)  \
1672   67               16.0   87.0       273.4                104.2   
349    31               27.0  118.0       145.5                105.4   
1939   24               19.0  118.0       145.5                104.9   
626    65               27.0   95.0       227.1                105.4   
2576   31               27.0  125.0       165.3                105.3   
3796   72               24.0   94.0       233.7                105.3   
6308   33               30.0  116.0       154.3                105.8   
3443   51                2.0   71.0       149.9                100.6   
2838   48                2.0   79.0   

/var/folders/zd/8r845kcj6nn1ym5f96t9hpw00000gn/T/ipykernel_37709/1198593573.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(male_train.groupby('dur_bin')['residual'].agg(['mean','std','count']))


In [39]:
# Duration*Weight 상호작용 + Duration^2*BPM 추가
X_male4 = np.column_stack([
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Age'],
    train.loc[male_mask2, 'Age'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'Weight(lb)'],
    train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'BPM'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Weight(lb)'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Temp_diff'] * train.loc[male_mask2, 'Age'],
    train.loc[male_mask2, 'Weight(lb)'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Age'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Weight(lb)'],
    height_total[male_mask2],
    height_total[male_mask2] * train.loc[male_mask2, 'Exercise_Duration'],
    height_total[male_mask2] * train.loc[male_mask2, 'BPM'],
    height_total[male_mask2] * train.loc[male_mask2, 'Weight(lb)'],
    # 추가
    train.loc[male_mask2, 'Exercise_Duration']**2 * train.loc[male_mask2, 'BPM'],
    train.loc[male_mask2, 'Exercise_Duration']**2 * train.loc[male_mask2, 'Weight(lb)'],
    train.loc[male_mask2, 'Exercise_Duration']**2 * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Weight(lb)'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'Weight(lb)'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Weight(lb)'] * train.loc[male_mask2, 'Temp_diff'],
])
c_male4, _, _, _ = lstsq(np.column_stack([X_male4, np.ones(len(X_male4))]),
                          train.loc[male_mask2, 'Calories_Burned'].values, rcond=None)
pred_male4 = X_male4 @ c_male4[:21] + c_male4[21]
print("남성 RMSE:", rmse(train.loc[male_mask2, 'Calories_Burned'].values, pred_male4))

res_male4 = pd.Series(train.loc[male_mask2, 'Calories_Burned'].values - pred_male4, index=train[male_mask2].index)
male_train['residual4'] = res_male4.values
print("\nDuration 구간별 잔차4 std:")
print(male_train.groupby('dur_bin')['residual4'].agg(['mean','std','count']))

print("\n잔차4 vs 주요 변수:")
for col in ['Age','Exercise_Duration','BPM','Weight(lb)','Temp_diff']:
    print(f"  {col}: {res_male4.corr(train.loc[male_mask2, col]):.4f}")
print(f"  Height: {res_male4.corr(height_total[male_mask2]):.4f}")

남성 RMSE: 1.1686955668935348

Duration 구간별 잔차4 std:
              mean       std  count
dur_bin                            
(0, 5]    0.063580  0.993672    545
(5, 10]  -0.066950  0.736801    655
(10, 15]  0.038472  0.831025    685
(15, 20] -0.025585  1.161966    611
(20, 25]  0.028240  1.422480    643
(25, 30] -0.033240  1.645926    592
(30, 35]       NaN       NaN      0

잔차4 vs 주요 변수:
  Age: 0.0000
  Exercise_Duration: -0.0072
  BPM: -0.0000
  Weight(lb): -0.0000
  Temp_diff: -0.0087
  Height: -0.0000


/var/folders/zd/8r845kcj6nn1ym5f96t9hpw00000gn/T/ipykernel_37709/3611403718.py:34: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(male_train.groupby('dur_bin')['residual4'].agg(['mean','std','count']))


In [40]:
# Duration^2, Duration^3 관련 항 집중 추가
X_male5 = np.column_stack([
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Age'],
    train.loc[male_mask2, 'Age'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'Weight(lb)'],
    train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'BPM'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Weight(lb)'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Temp_diff'] * train.loc[male_mask2, 'Age'],
    train.loc[male_mask2, 'Weight(lb)'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Age'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Weight(lb)'],
    height_total[male_mask2],
    height_total[male_mask2] * train.loc[male_mask2, 'Exercise_Duration'],
    height_total[male_mask2] * train.loc[male_mask2, 'BPM'],
    height_total[male_mask2] * train.loc[male_mask2, 'Weight(lb)'],
    train.loc[male_mask2, 'Exercise_Duration']**2 * train.loc[male_mask2, 'BPM'],
    train.loc[male_mask2, 'Exercise_Duration']**2 * train.loc[male_mask2, 'Weight(lb)'],
    train.loc[male_mask2, 'Exercise_Duration']**2 * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Weight(lb)'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'Weight(lb)'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Weight(lb)'] * train.loc[male_mask2, 'Temp_diff'],
    # Duration^3 항 추가
    train.loc[male_mask2, 'Exercise_Duration']**3 * train.loc[male_mask2, 'BPM'],
    train.loc[male_mask2, 'Exercise_Duration']**3 * train.loc[male_mask2, 'Weight(lb)'],
    train.loc[male_mask2, 'Exercise_Duration']**2 * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Exercise_Duration']**2 * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Age'],
    train.loc[male_mask2, 'Exercise_Duration']**2 * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Exercise_Duration']**2,
])
c_male5, _, _, _ = lstsq(np.column_stack([X_male5, np.ones(len(X_male5))]),
                          train.loc[male_mask2, 'Calories_Burned'].values, rcond=None)
pred_male5 = X_male5 @ c_male5[:27] + c_male5[27]
print("남성 RMSE:", rmse(train.loc[male_mask2, 'Calories_Burned'].values, pred_male5))

res_male5 = pd.Series(train.loc[male_mask2, 'Calories_Burned'].values - pred_male5, index=train[male_mask2].index)
male_train['residual5'] = res_male5.values
print("\nDuration 구간별 잔차5 std:")
print(male_train.groupby('dur_bin')['residual5'].agg(['mean','std','count']))

남성 RMSE: 0.7422902224192683

Duration 구간별 잔차5 std:
              mean       std  count
dur_bin                            
(0, 5]    0.053537  0.632256    545
(5, 10]  -0.094988  0.596803    655
(10, 15]  0.061007  0.659712    685
(15, 20]  0.009421  0.712229    611
(20, 25] -0.013437  0.812773    643
(25, 30] -0.009909  0.976787    592
(30, 35]       NaN       NaN      0


/var/folders/zd/8r845kcj6nn1ym5f96t9hpw00000gn/T/ipykernel_37709/3586873928.py:40: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(male_train.groupby('dur_bin')['residual5'].agg(['mean','std','count']))


In [41]:
X_male6 = np.column_stack([
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Age'],
    train.loc[male_mask2, 'Age'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'Weight(lb)'],
    train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'BPM'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Weight(lb)'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Temp_diff'] * train.loc[male_mask2, 'Age'],
    train.loc[male_mask2, 'Weight(lb)'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Age'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Weight(lb)'],
    height_total[male_mask2],
    height_total[male_mask2] * train.loc[male_mask2, 'Exercise_Duration'],
    height_total[male_mask2] * train.loc[male_mask2, 'BPM'],
    height_total[male_mask2] * train.loc[male_mask2, 'Weight(lb)'],
    train.loc[male_mask2, 'Exercise_Duration']**2 * train.loc[male_mask2, 'BPM'],
    train.loc[male_mask2, 'Exercise_Duration']**2 * train.loc[male_mask2, 'Weight(lb)'],
    train.loc[male_mask2, 'Exercise_Duration']**2 * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Weight(lb)'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'Weight(lb)'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Exercise_Duration'] * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Weight(lb)'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Exercise_Duration']**3 * train.loc[male_mask2, 'BPM'],
    train.loc[male_mask2, 'Exercise_Duration']**3 * train.loc[male_mask2, 'Weight(lb)'],
    train.loc[male_mask2, 'Exercise_Duration']**2 * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Exercise_Duration']**2 * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Age'],
    train.loc[male_mask2, 'Exercise_Duration']**2 * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Exercise_Duration']**2,
    # Duration^4 항 추가
    train.loc[male_mask2, 'Exercise_Duration']**4 * train.loc[male_mask2, 'BPM'],
    train.loc[male_mask2, 'Exercise_Duration']**4 * train.loc[male_mask2, 'Weight(lb)'],
    train.loc[male_mask2, 'Exercise_Duration']**3 * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Temp_diff'],
    train.loc[male_mask2, 'Exercise_Duration']**3 * train.loc[male_mask2, 'BPM'] * train.loc[male_mask2, 'Age'],
    train.loc[male_mask2, 'Exercise_Duration']**3,
    train.loc[male_mask2, 'Exercise_Duration']**4,
])
c_male6, _, _, _ = lstsq(np.column_stack([X_male6, np.ones(len(X_male6))]),
                          train.loc[male_mask2, 'Calories_Burned'].values, rcond=None)
pred_male6 = X_male6 @ c_male6[:33] + c_male6[33]
print("남성 RMSE:", rmse(train.loc[male_mask2, 'Calories_Burned'].values, pred_male6))

res_male6 = pd.Series(train.loc[male_mask2, 'Calories_Burned'].values - pred_male6, index=train[male_mask2].index)
male_train['residual6'] = res_male6.values
print("\nDuration 구간별 잔차6 std:")
print(male_train.groupby('dur_bin')['residual6'].agg(['mean','std','count']))

# 전체 합쳐서 RMSE 확인
pred_combined2 = np.zeros(len(train))
pred_combined2[male_mask2.values] = pred_male6
pred_combined2[female_mask2.values] = pred_female2
print("\n전체 RMSE (float):", rmse(train['Calories_Burned'].values, pred_combined2))
print("전체 RMSE (round):", rmse(train['Calories_Burned'].values, np.rint(np.clip(pred_combined2, 0, None))))

남성 RMSE: 0.6333017401754854

Duration 구간별 잔차6 std:
              mean       std  count
dur_bin                            
(0, 5]    0.012740  0.735708    545
(5, 10]  -0.051074  0.546539    655
(10, 15]  0.090201  0.519418    685
(15, 20] -0.066805  0.534130    611
(20, 25] -0.012894  0.646960    643
(25, 30]  0.023363  0.789936    592
(30, 35]       NaN       NaN      0

전체 RMSE (float): 0.6131662257453175
전체 RMSE (round): 0.663023378170031


/var/folders/zd/8r845kcj6nn1ym5f96t9hpw00000gn/T/ipykernel_37709/1336058259.py:45: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(male_train.groupby('dur_bin')['residual6'].agg(['mean','std','count']))


In [42]:
from sklearn.preprocessing import PolynomialFeatures
from numpy.linalg import lstsq

# 남성 핵심 변수만으로 고차 polynomial
male_df = train[male_mask2].copy()
male_df['Temp_diff'] = male_df['Body_Temperature(F)'] - 98.6
male_df['Height_Total'] = male_df['Height(Feet)'] * 12 + male_df['Height(Remainder_Inches)']

core_cols = ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'Weight(lb)', 'Height_Total']
X_core = male_df[core_cols].values

for deg in [3, 4, 5]:
    poly = PolynomialFeatures(degree=deg, include_bias=False)
    X_poly = poly.fit_transform(X_core)
    c, _, _, _ = lstsq(np.column_stack([X_poly, np.ones(len(X_poly))]),
                        male_df['Calories_Burned'].values, rcond=None)
    pred = X_poly @ c[:X_poly.shape[1]] + c[X_poly.shape[1]]
    res = pd.Series(male_df['Calories_Burned'].values - pred, index=male_df.index)
    male_df['res'] = res.values
    print(f"\ndeg={deg} 남성 RMSE: {rmse(male_df['Calories_Burned'].values, pred):.4f}")
    print(f"  피처 수: {X_poly.shape[1]}")
    print("  Duration 구간별 std:")
    male_df['dur_bin'] = pd.cut(male_df['Exercise_Duration'], bins=[0,5,10,15,20,25,30,35])
    print(male_df.groupby('dur_bin')['res'].std().values)


deg=3 남성 RMSE: 0.2873
  피처 수: 83
  Duration 구간별 std:
[0.28351898 0.29049606 0.28763315 0.28636615 0.27879626 0.29750693
        nan]

deg=4 남성 RMSE: 0.2829
  피처 수: 209
  Duration 구간별 std:
[0.2806926  0.28386844 0.28311097 0.28254479 0.27562028 0.29246114
        nan]

deg=5 남성 RMSE: 0.2751
  피처 수: 461
  Duration 구간별 std:
[0.2682581  0.27427895 0.27939164 0.27311462 0.27259015 0.28288686
        nan]


/var/folders/zd/8r845kcj6nn1ym5f96t9hpw00000gn/T/ipykernel_37709/2823403117.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(male_df.groupby('dur_bin')['res'].std().values)
/var/folders/zd/8r845kcj6nn1ym5f96t9hpw00000gn/T/ipykernel_37709/2823403117.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(male_df.groupby('dur_bin')['res'].std().values)
/var/folders/zd/8r845kcj6nn1ym5f96t9hpw00000gn/T/ipykernel_37709/2823403117.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from numpy.linalg import lstsq

# 공통 피처 준비 함수
def prep_df(df):
    d = df.copy()
    d['Temp_diff'] = d['Body_Temperature(F)'] - 98.6
    d['Height_Total'] = d['Height(Feet)'] * 12 + d['Height(Remainder_Inches)']
    return d

core_cols = ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'Weight(lb)', 'Height_Total'] # 중요 피처
DEGREE = 5  # 일단 5로  () 6개 변수의 모든 5차 조합 생성용)

train_prep = prep_df(train)
test_prep  = prep_df(test)

# 성별 분리
male_mask2   = train_prep['Gender'] == 'M'
female_mask2 = ~male_mask2
male_test    = test_prep['Gender'] == 'M'
female_test  = ~male_test

results = {}
for gender, mask, test_mask in [('M', male_mask2, male_test), ('F', female_mask2, female_test)]:
    X_tr = train_prep.loc[mask, core_cols].values
    y_tr = train_prep.loc[mask, 'Calories_Burned'].values
    X_te = test_prep.loc[test_mask, core_cols].values

    poly = PolynomialFeatures(degree=DEGREE, include_bias=False)
    X_tr_p = poly.fit_transform(X_tr)
    X_te_p = poly.transform(X_te)

    X_tr_b = np.column_stack([X_tr_p, np.ones(len(X_tr_p))])
    X_te_b = np.column_stack([X_te_p, np.ones(len(X_te_p))])

    c, _, _, _ = lstsq(X_tr_b, y_tr, rcond=None) # lstsq (최소제곱법) 으로 계수 학습 — Ridge 없이 순수 선형회귀
    pred_tr = X_tr_b @ c
    pred_te = X_te_b @ c

    print(f"Gender={gender} RMSE: {rmse(y_tr, pred_tr):.4f}  "
            f"round: {rmse(y_tr, np.rint(np.clip(pred_tr,0,None))):.4f}") 
    results[gender] = {'c': c, 'poly': poly, 'pred_tr': pred_tr, 'pred_te': pred_te,
                        'mask': mask, 'test_mask': test_mask}

# 전체 OOF RMSE
pred_all = np.zeros(len(train))
pred_all[male_mask2.values]   = results['M']['pred_tr']
pred_all[female_mask2.values] = results['F']['pred_tr']
print(f"\n전체 RMSE float: {rmse(train['Calories_Burned'].values, pred_all):.4f}")
print(f"전체 RMSE round: {rmse(train['Calories_Burned'].values, np.rint(np.clip(pred_all,0,None))):.4f}")

# 제출 파일
test_pred_all = np.zeros(len(test))
test_pred_all[male_test.values]   = results['M']['pred_te']
test_pred_all[female_test.values] = results['F']['pred_te']
test_pred_round = np.rint(np.clip(test_pred_all, 0, None)).astype(int) # test 예측 후 반올림해서 저장용

sub = pd.read_csv("sample_submission.csv")
sub['Calories_Burned'] = test_pred_round
sub.to_csv("submit_poly5_gender.csv", index=False)
print("\n저장: submit_poly5_gender.csv")
print(f"test 예측 mean/min/max: {test_pred_round.mean():.1f} / {test_pred_round.min()} / {test_pred_round.max()}")

Gender=M RMSE: 0.2751  round: 0.1717
Gender=F RMSE: 0.2726  round: 0.1716

전체 RMSE float: 0.2738
전체 RMSE round: 0.1717

저장: submit_poly5_gender.csv
test 예측 mean/min/max: 89.7 / 1 / 314


In [55]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import RidgeCV
import numpy as np

def prep_df(df):
    d = df.copy()
    d['Temp_diff'] = d['Body_Temperature(F)'] - 98.6
    d['Height_Total'] = d['Height(Feet)'] * 12 + d['Height(Remainder_Inches)']
    return d

core_cols = ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'Weight(lb)', 'Height_Total']

train_prep = prep_df(train)
test_prep  = prep_df(test)

male_mask2   = train_prep['Gender'] == 'M'
female_mask2 = ~male_mask2
male_test    = test_prep['Gender'] == 'M'
female_test  = ~male_test

pred_all      = np.zeros(len(train))
test_pred_all = np.zeros(len(test))

for gender, mask, test_mask in [('M', male_mask2, male_test), ('F', female_mask2, female_test)]:
    X_tr = train_prep.loc[mask, core_cols].values
    y_tr = train_prep.loc[mask, 'Calories_Burned'].values
    X_te = test_prep.loc[test_mask, core_cols].values

    poly = PolynomialFeatures(degree=5, include_bias=False)
    X_tr_p = poly.fit_transform(X_tr)
    X_te_p = poly.transform(X_te)          # train fit → test transform only

    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr_p)
    X_te_s = sc.transform(X_te_p)          # train fit → test transform only

    model = RidgeCV(alphas=np.logspace(-4, 2, 100))
    model.fit(X_tr_s, y_tr)
    print(f"Gender={gender}  best alpha: {model.alpha_:.5f}  "
            f"train RMSE round: {rmse(y_tr, np.rint(np.clip(model.predict(X_tr_s),0,None))):.4f}")

    pred_all[mask.values]          = model.predict(X_tr_s)
    test_pred_all[test_mask.values] = model.predict(X_te_s)

print(f"\n전체 train RMSE round: {rmse(train['Calories_Burned'].values, np.rint(np.clip(pred_all,0,None))):.4f}")

test_pred_round = np.rint(np.clip(test_pred_all, 0, None)).astype(int)
sub = pd.read_csv("sample_submission.csv")
sub['Calories_Burned'] = test_pred_round
sub.to_csv("submit_ridge_poly5_final.csv", index=False)
print("저장: submit_ridge_poly5_final.csv")

Gender=M  best alpha: 0.00081  train RMSE round: 0.1596
Gender=F  best alpha: 0.00327  train RMSE round: 0.1537

전체 train RMSE round: 0.1566
저장: submit_ridge_poly5_final.csv


# RMSE 0.1566 리더보드 0.2019900988

In [56]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for gender, mask in [('M', male_mask2), ('F', female_mask2)]:
    print(f"\n=== Gender={gender} ===")
    X_g = train_prep.loc[mask, core_cols].values
    y_g = train_prep.loc[mask, 'Calories_Burned'].values
    
    for deg in [5, 6, 7, 8, 9, 10]:
        oof_g = np.zeros(len(y_g))
        for tr_idx, va_idx in kf.split(X_g):
            poly = PolynomialFeatures(degree=deg, include_bias=False)
            X_tr_p = poly.fit_transform(X_g[tr_idx])
            X_va_p = poly.transform(X_g[va_idx])
            sc = StandardScaler()
            X_tr_s = sc.fit_transform(X_tr_p)
            X_va_s = sc.transform(X_va_p)
            model = RidgeCV(alphas=np.logspace(-4, 3, 200))
            model.fit(X_tr_s, y_g[tr_idx])
            oof_g[va_idx] = model.predict(X_va_s)
        
        r = rmse(y_g, np.rint(np.clip(oof_g, 0, None)))
        print(f"  deg={deg}  OOF round: {r:.4f}")


=== Gender=M ===
  deg=5  OOF round: 0.2350
  deg=6  OOF round: 0.2461
  deg=7  OOF round: 0.2515
  deg=8  OOF round: 0.2630
  deg=9  OOF round: 0.2744
  deg=10  OOF round: 0.2915

=== Gender=F ===
  deg=5  OOF round: 0.2149
  deg=6  OOF round: 0.2257
  deg=7  OOF round: 0.2269
  deg=8  OOF round: 0.2332
  deg=9  OOF round: 0.2465
  deg=10  OOF round: 0.2518


In [57]:
# 변수 조합별 OOF 성능 비교
from itertools import combinations

base_cols = ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'Weight(lb)', 'Height_Total']

# Height_Total 빼보기, BMI 추가해보기
train_prep['BMI'] = 703 * train_prep['Weight(lb)'] / (train_prep['Height_Total'] ** 2)
test_prep['BMI']  = 703 * test_prep['Weight(lb)']  / (test_prep['Height_Total'] ** 2)

col_sets = {
    '6개(현재)':   ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'Weight(lb)', 'Height_Total'],
    'Height제외':  ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'Weight(lb)'],
    'BMI추가':     ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'Weight(lb)', 'Height_Total', 'BMI'],
    'BMI로대체':   ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'BMI'],
}

for name, cols in col_sets.items():
    oof_all = np.zeros(len(train))
    for gender, mask in [('M', male_mask2), ('F', female_mask2)]:
        X_g = train_prep.loc[mask, cols].values
        y_g = train_prep.loc[mask, 'Calories_Burned'].values
        oof_g = np.zeros(len(y_g))
        for tr_idx, va_idx in kf.split(X_g):
            poly = PolynomialFeatures(degree=5, include_bias=False)
            X_tr_p = poly.fit_transform(X_g[tr_idx])
            X_va_p = poly.transform(X_g[va_idx])
            sc = StandardScaler()
            X_tr_s = sc.fit_transform(X_tr_p)
            X_va_s = sc.transform(X_va_p)
            model = RidgeCV(alphas=np.logspace(-4, 3, 200))
            model.fit(X_tr_s, y_g[tr_idx])
            oof_g[va_idx] = model.predict(X_va_s)
        oof_all[mask.values] = oof_g
    r = rmse(train['Calories_Burned'].values, np.rint(np.clip(oof_all, 0, None)))
    print(f"{name}: OOF round {r:.4f}")

6개(현재): OOF round 0.2251
Height제외: OOF round 0.2126
BMI추가: OOF round 0.2272
BMI로대체: OOF round 2.9249


In [58]:
best_cols = ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'Weight(lb)']

pred_all      = np.zeros(len(train))
test_pred_all = np.zeros(len(test))

for gender, mask, test_mask in [('M', male_mask2, male_test), ('F', female_mask2, female_test)]:
    X_tr = train_prep.loc[mask, best_cols].values
    y_tr = train_prep.loc[mask, 'Calories_Burned'].values
    X_te = test_prep.loc[test_mask, best_cols].values

    poly = PolynomialFeatures(degree=5, include_bias=False)
    X_tr_p = poly.fit_transform(X_tr)
    X_te_p = poly.transform(X_te)

    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr_p)
    X_te_s = sc.transform(X_te_p)

    model = RidgeCV(alphas=np.logspace(-4, 3, 200))
    model.fit(X_tr_s, y_tr)
    print(f"Gender={gender}  alpha: {model.alpha_:.5f}  "
          f"train round: {rmse(y_tr, np.rint(np.clip(model.predict(X_tr_s),0,None))):.4f}")

    pred_all[mask.values]           = model.predict(X_tr_s)
    test_pred_all[test_mask.values] = model.predict(X_te_s)

print(f"\n전체 train RMSE round: {rmse(train['Calories_Burned'].values, np.rint(np.clip(pred_all,0,None))):.4f}")

test_pred_round = np.rint(np.clip(test_pred_all, 0, None)).astype(int)
sub = pd.read_csv("sample_submission.csv")
sub['Calories_Burned'] = test_pred_round
sub.to_csv("submit_ridge_5cols.csv", index=False)
print("저장: submit_ridge_5cols.csv")

Gender=M  alpha: 0.00051  train round: 0.1527
Gender=F  alpha: 0.00170  train round: 0.1554

전체 train RMSE round: 0.1541
저장: submit_ridge_5cols.csv


In [64]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def prep_df(df):
    d = df.copy()
    d['Temp_diff'] = d['Body_Temperature(F)'] - 98.6
    d['Height_Total'] = d['Height(Feet)'] * 12 + d['Height(Remainder_Inches)']
    return d

# 🔹 core-only 컬럼 (Height 제외 버전이 좋았다고 했으니 일단 제외)
core_cols = ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'Weight(lb)']

train_prep = prep_df(train)
test_prep  = prep_df(test)

y_all = train_prep['Calories_Burned'].values.astype(float)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

degree_grid = [2,3,4,5,6]
alpha_grid  = np.logspace(-4, 1, 10)

best_score = 1e18
best_params = None

for DEGREE in degree_grid:
    for ALPHA in alpha_grid:

        oof_all = np.zeros(len(train_prep))

        for gender in ['M', 'F']:
            tr_mask = (train_prep['Gender'] == gender).values
            X_full = train_prep.loc[tr_mask, core_cols].values
            y_full = y_all[tr_mask]

            oof_g = np.zeros(len(y_full))

            for tr_idx, va_idx in kf.split(X_full):
                X_tr = X_full[tr_idx]
                X_va = X_full[va_idx]
                y_tr = y_full[tr_idx]

                poly = PolynomialFeatures(degree=DEGREE, include_bias=False)
                X_tr_p = poly.fit_transform(X_tr)
                X_va_p = poly.transform(X_va)

                sc = StandardScaler()
                X_tr_s = sc.fit_transform(X_tr_p)
                X_va_s = sc.transform(X_va_p)

                model = Ridge(alpha=ALPHA)
                model.fit(X_tr_s, y_tr)

                oof_g[va_idx] = model.predict(X_va_s)

            oof_all[tr_mask] = oof_g

        score = rmse(y_all, np.rint(np.clip(oof_all,0,None)))

        if score < best_score:
            best_score = score
            best_params = {"degree": DEGREE, "alpha": float(ALPHA)}

        print(f"deg={DEGREE} alpha={ALPHA:.5f} → {score:.4f}")

print("\n🔥 BEST:", best_params, "| OOF round RMSE:", best_score)

deg=2 alpha=0.00010 → 0.1361
deg=2 alpha=0.00036 → 0.1356
deg=2 alpha=0.00129 → 0.1332
deg=2 alpha=0.00464 → 0.1558
deg=2 alpha=0.01668 → 0.2295
deg=2 alpha=0.05995 → 0.3940
deg=2 alpha=0.21544 → 0.6429
deg=2 alpha=0.77426 → 0.9528
deg=2 alpha=2.78256 → 1.5225
deg=2 alpha=10.00000 → 2.7098
deg=3 alpha=0.00010 → 0.1740
deg=3 alpha=0.00036 → 0.1785
deg=3 alpha=0.00129 → 0.1880
deg=3 alpha=0.00464 → 0.2139
deg=3 alpha=0.01668 → 0.2733
deg=3 alpha=0.05995 → 0.3738
deg=3 alpha=0.21544 → 0.4737
deg=3 alpha=0.77426 → 0.5948
deg=3 alpha=2.78256 → 0.7718
deg=3 alpha=10.00000 → 1.0863
deg=4 alpha=0.00010 → 0.2036
deg=4 alpha=0.00036 → 0.2020
deg=4 alpha=0.00129 → 0.1993
deg=4 alpha=0.00464 → 0.2066
deg=4 alpha=0.01668 → 0.2269
deg=4 alpha=0.05995 → 0.2917
deg=4 alpha=0.21544 → 0.4128
deg=4 alpha=0.77426 → 0.5715
deg=4 alpha=2.78256 → 0.7398
deg=4 alpha=10.00000 → 0.9206
deg=5 alpha=0.00010 → 0.2163
deg=5 alpha=0.00036 → 0.2117
deg=5 alpha=0.00129 → 0.2135
deg=5 alpha=0.00464 → 0.2251
deg=5 alpha

지금 상황 정리

core-only +
gender 분리 +
degree=2 +
alpha≈0.0013

→ 0.133 (OOF round)

이 문제는 고차 다항식이 아니라
핵심 물리식 구조 + 2차 상호작용이면 충분했다.

In [61]:
print("OOF float RMSE:",
      rmse(y_all, np.clip(oof_all,0,None)))

OOF float RMSE: 0.7372703276014538


In [62]:
print("pct 300:", np.mean(test_pred_round == 300))
print("pct 1:", np.mean(test_pred_round == 1))
print("unique preds:", len(np.unique(test_pred_round)))

pct 300: 0.0
pct 1: 0.0010666666666666667
unique preds: 261


# 어ㅏㅏㅗ리ㅏㅑㅇ노리ㅏㅑㄴ몰;ㅑ농;랴ㅗㅁㅇㄴ

In [69]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# =========================
# 0) 설정
# =========================
RANDOM_STATE = 42
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

DEGREE = 3
ALPHA_SQ = 0.009

TOP5_FIXED = ['Log_Weight_BPM_Dur', 'Effort', 'Weight_x_Duration', 'Log_Duration', 'Intensity']

alpha_grid = np.linspace(0.02, 0.10, 41)      # alpha_id 후보
w_grid     = np.linspace(0.95, 1.00, 101)     # weight 후보
eps_grid   = np.linspace(0.0, 0.010, 51)      # rule-adjust eps 후보

# =========================
# 1) 원본에서 다시 시작 (상태 꼬임 방지)
# =========================
train_x_raw = train.drop(['ID', 'Calories_Burned'], axis=1).copy()
train_y = train['Calories_Burned'].astype(float).copy()
test_x_raw  = test.drop(['ID'], axis=1).copy()

before_cols = list(train_x_raw.columns)

train_x_feat = create_features_safe(train_x_raw)
test_x_feat  = create_features_safe(test_x_raw)

generated_cols = [c for c in train_x_feat.columns if c not in before_cols]

# =========================
# 2) 원핫 인코딩 (train fit -> test transform)
# =========================
cat_cols = [c for c in ['Gender', 'Weight_Status'] if c in train_x_feat.columns]
if len(cat_cols) > 0:
    enc = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
    enc.fit(train_x_feat[cat_cols])

    tr_cat = enc.transform(train_x_feat[cat_cols])
    te_cat = enc.transform(test_x_feat[cat_cols])
    cat_names = enc.get_feature_names_out(cat_cols)

    train_x_feat = train_x_feat.drop(columns=cat_cols)
    test_x_feat  = test_x_feat.drop(columns=cat_cols)

    train_x_feat[cat_names] = tr_cat
    test_x_feat[cat_names]  = te_cat

train_x_final = train_x_feat.copy()
test_x_final  = test_x_feat.copy()

# =========================
# 3) (중요) keep_cols 안전 구성: 존재하는 컬럼만 사용
# =========================
# base_feats = 파생변수 제외한 "원본+원핫"만
base_feats = [c for c in train_x_final.columns if c not in generated_cols]

# TOP5 중 실제 존재하는 것만
top5_exists = [c for c in TOP5_FIXED if c in train_x_final.columns]
missing_top5 = [c for c in TOP5_FIXED if c not in train_x_final.columns]

print("TOP5 exists:", top5_exists)
print("TOP5 missing:", missing_top5)

keep_cols_raw = base_feats + top5_exists
keep_cols = [c for c in keep_cols_raw if c in train_x_final.columns]

# train/test 컬럼 정렬 (혹시 test에 없는 컬럼 있으면 0 채움)
train_red = train_x_final.reindex(columns=keep_cols, fill_value=0).copy()
test_red  = test_x_final.reindex(columns=keep_cols,  fill_value=0).copy()
train_red, test_red = train_red.align(test_red, join="left", axis=1, fill_value=0)

y = train_y.values.astype(float)
y_sqrt = np.sqrt(np.clip(y, 0, None))

print("Final train/test shape:", train_x_final.shape, test_x_final.shape)
print("Reduced train/test shape:", train_red.shape, test_red.shape)

# =========================
# 4) fold cache (poly+scaler)
# =========================
fold_cache = []
for tr_idx, va_idx in kf.split(train_red):
    X_tr_raw = train_red.iloc[tr_idx]
    X_va_raw = train_red.iloc[va_idx]

    poly = PolynomialFeatures(degree=DEGREE, include_bias=False)
    X_tr = poly.fit_transform(X_tr_raw)
    X_va = poly.transform(X_va_raw)
    X_te = poly.transform(test_red)

    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_va_s = sc.transform(X_va)
    X_te_s = sc.transform(X_te)

    fold_cache.append((tr_idx, va_idx, X_tr_s, X_va_s, X_te_s))

# =========================
# 5) rule-adjust (경계 근처 tie-break)
# =========================
def rule_adjust(pred_float, ref_float, eps):
    pred_float = np.clip(pred_float, 0, None)
    ref_float = np.clip(ref_float, 0, None)

    lo = np.floor(pred_float)
    hi = lo + 1
    frac = pred_float - lo

    out = np.rint(pred_float)
    mask = np.abs(frac - 0.5) <= eps

    choose_hi = np.abs(ref_float - hi) < np.abs(ref_float - lo)
    out[mask] = np.where(choose_hi[mask], hi[mask], lo[mask])
    return np.clip(out, 0, None)

# =========================
# 6) 튜닝: alpha_id -> w -> eps (OOF round RMSE 최소화)
# =========================
best = {"rmse": 1e18, "alpha_id": None, "w": None, "eps": None}
best_artifacts = None

for alpha_id in tqdm(alpha_grid, desc="alpha_id search"):
    id_oof = np.zeros(len(train_red))
    id_test = np.zeros(len(test_red))
    sq_oof = np.zeros(len(train_red))
    sq_test = np.zeros(len(test_red))

    for tr_idx, va_idx, X_tr_s, X_va_s, X_te_s in fold_cache:
        # identity
        m_id = Ridge(alpha=float(alpha_id), random_state=RANDOM_STATE)
        m_id.fit(X_tr_s, y[tr_idx])
        id_oof[va_idx] = m_id.predict(X_va_s)
        id_test += m_id.predict(X_te_s) / N_SPLITS

        # sqrt
        m_sq = Ridge(alpha=float(ALPHA_SQ), random_state=RANDOM_STATE)
        m_sq.fit(X_tr_s, y_sqrt[tr_idx])
        pred_va_sq = m_sq.predict(X_va_s)
        pred_te_sq = m_sq.predict(X_te_s)
        sq_oof[va_idx] = np.clip(pred_va_sq, 0, None) ** 2
        sq_test += (np.clip(pred_te_sq, 0, None) ** 2) / N_SPLITS

    id_oof = np.clip(id_oof, 0, None)
    sq_oof = np.clip(sq_oof, 0, None)
    id_test = np.clip(id_test, 0, None)
    sq_test = np.clip(sq_test, 0, None)

    for w in w_grid:
        final_oof = np.clip(w * id_oof + (1 - w) * sq_oof, 0, None)

        # eps 탐색
        for eps in eps_grid:
            oof_int = rule_adjust(final_oof, sq_oof, eps)
            score = rmse(y, oof_int)
            if score < best["rmse"]:
                best = {"rmse": float(score), "alpha_id": float(alpha_id), "w": float(w), "eps": float(eps)}
                best_artifacts = (id_oof.copy(), sq_oof.copy(), id_test.copy(), sq_test.copy())

print("\n✅ BEST params:", best)

# =========================
# 7) BEST로 test 예측 + 저장
# =========================
id_oof, sq_oof, id_test, sq_test = best_artifacts
w = best["w"]
eps = best["eps"]

final_oof = np.clip(w * id_oof + (1 - w) * sq_oof, 0, None)
final_test = np.clip(w * id_test + (1 - w) * sq_test, 0, None)

oof_int = rule_adjust(final_oof, sq_oof, eps)
test_int = rule_adjust(final_test, sq_test, eps)

print("OOF RMSE (rule-adjust):", rmse(y, oof_int))
print("Test stats mean/min/max:", float(test_int.mean()), float(test_int.min()), float(test_int.max()))

sub = pd.read_csv("sample_submission.csv")
sub["Calories_Burned"] = test_int.astype(int)
out_name = "submit_top5_retuned_rule_safe.csv"
sub.to_csv(out_name, index=False)
print("✅ saved:", out_name)

TOP5 exists: ['Log_Weight_BPM_Dur', 'Effort', 'Weight_x_Duration', 'Log_Duration', 'Intensity']
TOP5 missing: []
Final train/test shape: (7500, 34) (7500, 28)
Reduced train/test shape: (7500, 22) (7500, 22)


alpha_id search: 100%|██████████| 41/41 [01:46<00:00,  2.60s/it]


✅ BEST params: {'rmse': 0.21878452108562588, 'alpha_id': 0.078, 'w': 0.9875, 'eps': 0.0014}
OOF RMSE (rule-adjust): 0.21878452108562588
Test stats mean/min/max: 18.612133333333333 0.0 61.0
✅ saved: submit_top5_retuned_rule_safe.csv


정수 타겟 특성을 반영해 라운딩을 적용했고, 이후 0.5 경계 근처(±0.0014)에서만 보조 예측으로 tie-break하는 규칙 기반 보정으로 0.2188까지 개선

## rule-adjust (eps=0.004): 0.21848 여기서 더 강화해볼건 없어?

In [54]:
# test 누수 없는지 확인
# PolynomialFeatures: train만으로 fit → test는 transform만 ✅
# StandardScaler 썼다면: train만으로 fit → test는 transform만 ✅

# 아까 deg=5 lstsq 버전은 StandardScaler 없었으니 문제없음
# 바로 제출 가능한 코드 재실행
for gender, mask, test_mask in [('M', male_mask2, male_test), ('F', female_mask2, female_test)]:
    X_tr = train_prep.loc[mask, core_cols].values
    y_tr = train_prep.loc[mask, 'Calories_Burned'].values
    X_te = test_prep.loc[test_mask, core_cols].values

    poly = PolynomialFeatures(degree=5, include_bias=False)
    X_tr_p = poly.fit_transform(X_tr)   # train만 fit
    X_te_p = poly.transform(X_te)       # test는 transform만

    X_tr_b = np.column_stack([X_tr_p, np.ones(len(X_tr_p))])
    X_te_b = np.column_stack([X_te_p, np.ones(len(X_te_p))])

    c, _, _, _ = lstsq(X_tr_b, y_tr, rcond=None)
    test_pred_all[test_mask.values] = X_te_b @ c

test_pred_round = np.rint(np.clip(test_pred_all, 0, None)).astype(int)
sub = pd.read_csv("sample_submission.csv")
sub['Calories_Burned'] = test_pred_round
sub.to_csv("submit_poly5_final.csv", index=False)
print("저장 완료!")
print(f"train RMSE round: {rmse(train['Calories_Burned'].values, np.rint(np.clip(pred_all,0,None))):.4f}")

저장 완료!
train RMSE round: 0.1717


## 0.17 !?!?!?

In [ ]:
for deg in [6, 7, 8]:
    results_deg = {}
    for gender, mask, test_mask in [('M', male_mask2, male_test), ('F', female_mask2, female_test)]:
        X_tr = train_prep.loc[mask, core_cols].values
        y_tr = train_prep.loc[mask, 'Calories_Burned'].values

        poly = PolynomialFeatures(degree=deg, include_bias=False)
        X_tr_p = poly.fit_transform(X_tr)
        X_tr_b = np.column_stack([X_tr_p, np.ones(len(X_tr_p))])
        c, _, _, _ = lstsq(X_tr_b, y_tr, rcond=None)
        pred_tr = X_tr_b @ c
        results_deg[gender] = pred_tr

    pred_all = np.zeros(len(train))
    pred_all[male_mask2.values]   = results_deg['M']
    pred_all[female_mask2.values] = results_deg['F']
    print(f"deg={deg}  float: {rmse(train['Calories_Burned'].values, pred_all):.4f}  "
            f"round: {rmse(train['Calories_Burned'].values, np.rint(np.clip(pred_all,0,None))):.4f}  "
            f"피처수: {X_tr_p.shape[1]}")

deg=6  float: 0.2639  round: 0.1657  피처수: 923
deg=7  float: 0.2537  round: 0.1665  피처수: 1715
deg=8  float: 0.2467  round: 0.1617  피처수: 3002


In [45]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for deg in [9, 10, 12]:
    oof_all = np.zeros(len(train))
    
    for gender, mask in [('M', male_mask2), ('F', female_mask2)]:
        X_g = train_prep.loc[mask, core_cols].values
        y_g = train_prep.loc[mask, 'Calories_Burned'].values
        idx_g = np.where(mask.values)[0]
        oof_g = np.zeros(len(y_g))
        
        for tr_idx, va_idx in kf.split(X_g):
            poly = PolynomialFeatures(degree=deg, include_bias=False)
            X_tr_p = poly.fit_transform(X_g[tr_idx])
            X_va_p = poly.transform(X_g[va_idx])
            
            X_tr_b = np.column_stack([X_tr_p, np.ones(len(X_tr_p))])
            X_va_b = np.column_stack([X_va_p, np.ones(len(X_va_p))])
            
            c, _, _, _ = lstsq(X_tr_b, y_g[tr_idx], rcond=None)
            oof_g[va_idx] = X_va_b @ c
        
        oof_all[idx_g] = oof_g
    
    print(f"deg={deg}  OOF float: {rmse(train['Calories_Burned'].values, oof_all):.4f}  "
            f"OOF round: {rmse(train['Calories_Burned'].values, np.rint(np.clip(oof_all,0,None))):.4f}")

deg=9  OOF float: 3.8605  OOF round: 3.5554
deg=10  OOF float: 5.7009  OOF round: 4.6196
deg=12  OOF float: 9.8219  OOF round: 7.8059


In [46]:
for deg in [5, 6, 7, 8]:
    oof_all = np.zeros(len(train))
    
    for gender, mask in [('M', male_mask2), ('F', female_mask2)]:
        X_g = train_prep.loc[mask, core_cols].values
        y_g = train_prep.loc[mask, 'Calories_Burned'].values
        idx_g = np.where(mask.values)[0]
        oof_g = np.zeros(len(y_g))
        
        for tr_idx, va_idx in kf.split(X_g):
            poly = PolynomialFeatures(degree=deg, include_bias=False)
            X_tr_p = poly.fit_transform(X_g[tr_idx])
            X_va_p = poly.transform(X_g[va_idx])
            
            X_tr_b = np.column_stack([X_tr_p, np.ones(len(X_tr_p))])
            X_va_b = np.column_stack([X_va_p, np.ones(len(X_va_p))])
            
            c, _, _, _ = lstsq(X_tr_b, y_g[tr_idx], rcond=None)
            oof_g[va_idx] = X_va_b @ c
        
        oof_all[idx_g] = oof_g
    
    print(f"deg={deg}  OOF float: {rmse(train['Calories_Burned'].values, oof_all):.4f}  "
          f"OOF round: {rmse(train['Calories_Burned'].values, np.rint(np.clip(oof_all,0,None))):.4f}")

deg=5  OOF float: 0.3475  OOF round: 0.3219
deg=6  OOF float: 0.4758  OOF round: 0.4772
deg=7  OOF float: 1.2684  OOF round: 0.7672
deg=8  OOF float: 2.4044  OOF round: 2.2357


In [47]:
# Ridge 규제로 고차항을 안정화
from sklearn.linear_model import Ridge, RidgeCV

for deg in [5, 6, 7, 8]:
    oof_all = np.zeros(len(train))
    
    for gender, mask in [('M', male_mask2), ('F', female_mask2)]:
        X_g = train_prep.loc[mask, core_cols].values
        y_g = train_prep.loc[mask, 'Calories_Burned'].values
        idx_g = np.where(mask.values)[0]
        oof_g = np.zeros(len(y_g))
        
        for tr_idx, va_idx in kf.split(X_g):
            poly = PolynomialFeatures(degree=deg, include_bias=False)
            X_tr_p = poly.fit_transform(X_g[tr_idx])
            X_va_p = poly.transform(X_g[va_idx])
            
            from sklearn.preprocessing import StandardScaler
            sc = StandardScaler()
            X_tr_s = sc.fit_transform(X_tr_p)
            X_va_s = sc.transform(X_va_p)
            
            # RidgeCV로 최적 alpha 자동 탐색
            model = RidgeCV(alphas=[0.001, 0.01, 0.1, 1, 10, 100, 1000])
            model.fit(X_tr_s, y_g[tr_idx])
            oof_g[va_idx] = model.predict(X_va_s)
        
        oof_all[idx_g] = oof_g
    
    print(f"deg={deg}  OOF float: {rmse(train['Calories_Burned'].values, oof_all):.4f}  "
            f"OOF round: {rmse(train['Calories_Burned'].values, np.rint(np.clip(oof_all,0,None))):.4f}")

deg=5  OOF float: 0.2995  OOF round: 0.2277
deg=6  OOF float: 0.3023  OOF round: 0.2355
deg=7  OOF float: 0.3070  OOF round: 0.2439
deg=8  OOF float: 0.3074  OOF round: 0.2506


In [49]:
from sklearn.linear_model import RidgeCV

for deg in [5, 6]:
    oof_all = np.zeros(len(train))
    
    for gender, mask in [('M', male_mask2), ('F', female_mask2)]:
        X_g = train_prep.loc[mask, core_cols].values
        y_g = train_prep.loc[mask, 'Calories_Burned'].values
        idx_g = np.where(mask.values)[0]
        oof_g = np.zeros(len(y_g))
        
        for tr_idx, va_idx in kf.split(X_g):
            poly = PolynomialFeatures(degree=deg, include_bias=False)
            X_tr_p = poly.fit_transform(X_g[tr_idx])
            X_va_p = poly.transform(X_g[va_idx])
            
            sc = StandardScaler()
            X_tr_s = sc.fit_transform(X_tr_p)
            X_va_s = sc.transform(X_va_p)
            
            # alpha 범위 세밀하게
            alphas = np.logspace(-4, 2, 100)
            model = RidgeCV(alphas=alphas)
            model.fit(X_tr_s, y_g[tr_idx])
            oof_g[va_idx] = model.predict(X_va_s)
        
        oof_all[idx_g] = oof_g
    
    r_float = rmse(train['Calories_Burned'].values, oof_all)
    r_round = rmse(train['Calories_Burned'].values, np.rint(np.clip(oof_all, 0, None)))
    print(f"deg={deg}  OOF float: {r_float:.4f}  OOF round: {r_round:.4f}")

# 동시에 전체 데이터로 학습 후 test 예측 (deg=5)
test_pred_all = np.zeros(len(test))
for gender, mask, test_mask in [('M', male_mask2, male_test), ('F', female_mask2, female_test)]:
    X_tr = train_prep.loc[mask, core_cols].values
    y_tr = train_prep.loc[mask, 'Calories_Burned'].values
    X_te = test_prep.loc[test_mask, core_cols].values

    poly = PolynomialFeatures(degree=5, include_bias=False)
    X_tr_p = poly.fit_transform(X_tr)
    X_te_p = poly.transform(X_te)

    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr_p)
    X_te_s = sc.transform(X_te_p)

    alphas = np.logspace(-4, 2, 100)
    model = RidgeCV(alphas=alphas)
    model.fit(X_tr_s, y_tr)
    print(f"  Gender={gender} best alpha: {model.alpha_:.6f}")
    test_pred_all[test_mask.values] = model.predict(X_te_s)

test_pred_round = np.rint(np.clip(test_pred_all, 0, None)).astype(int)
sub = pd.read_csv("sample_submission.csv")
sub['Calories_Burned'] = test_pred_round
sub.to_csv("submit_ridge_poly5.csv", index=False)
print("\n저장: submit_ridge_poly5.csv")

deg=5  OOF float: 0.2996  OOF round: 0.2254
deg=6  OOF float: 0.3021  OOF round: 0.2361
  Gender=M best alpha: 0.000811
  Gender=F best alpha: 0.003275

저장: submit_ridge_poly5.csv


In [50]:
# 칼로리를 log/sqrt 변환했을 때 선형 관계가 더 잘 맞는지 확인
for gender, mask in [('M', male_mask2), ('F', female_mask2)]:
    X_g = train_prep.loc[mask, core_cols].values
    y_g = train_prep.loc[mask, 'Calories_Burned'].values
    y_log = np.log1p(y_g)
    y_sqrt = np.sqrt(y_g)

    poly = PolynomialFeatures(degree=5, include_bias=False)
    X_p = poly.fit_transform(X_g)
    X_b = np.column_stack([X_p, np.ones(len(X_p))])

    for y_name, y_trans, inv_fn in [
        ('원본',  y_g,    lambda x: x),
        ('log1p', y_log,  np.expm1),
        ('sqrt',  y_sqrt, lambda x: x**2),
    ]:
        c, _, _, _ = lstsq(X_b, y_trans, rcond=None)
        pred = inv_fn(X_b @ c)
        print(f"Gender={gender} y={y_name}  RMSE: {rmse(y_g, pred):.4f}")
    print()

Gender=M y=원본  RMSE: 0.2751
Gender=M y=log1p  RMSE: 0.6991
Gender=M y=sqrt  RMSE: 0.2932

Gender=F y=원본  RMSE: 0.2726
Gender=F y=log1p  RMSE: 0.6604
Gender=F y=sqrt  RMSE: 0.2923



In [51]:
# 데이터 타입과 소수점 패턴 확인
print("=== Duration 소수점 분포 ===")
dur_frac = train['Exercise_Duration'] % 1
print(dur_frac.value_counts().head(10))

print("\n=== BPM 소수점 분포 ===")
bpm_frac = train['BPM'] % 1
print(bpm_frac.value_counts().head(10))

print("\n=== Weight 소수점 분포 ===")
w_frac = (train['Weight(lb)'] * 10) % 1
print(w_frac.value_counts().head(5))

print("\n=== Body_Temperature 소수점 분포 ===")
temp_frac = (train['Body_Temperature(F)'] * 10) % 1
print(temp_frac.value_counts().head(5))

# 남성 잔차가 큰 샘플의 Duration 패턴
male_df2 = train[male_mask2].copy()
male_df2['pred'] = pred_male6  # 이전 최선 예측
male_df2['res'] = male_df2['Calories_Burned'] - male_df2['pred']
print("\n=== 잔차 큰 샘플 Duration 값 ===")
print(male_df2.nlargest(10, 'res')[['Exercise_Duration', 'BPM', 'Weight(lb)', 'Body_Temperature(F)', 'Calories_Burned', 'res']])

=== Duration 소수점 분포 ===
Exercise_Duration
0.0    7500
Name: count, dtype: int64

=== BPM 소수점 분포 ===
BPM
0.0    7500
Name: count, dtype: int64

=== Weight 소수점 분포 ===
Weight(lb)
0.0    7500
Name: count, dtype: int64

=== Body_Temperature 소수점 분포 ===
Body_Temperature(F)
0.0    7500
Name: count, dtype: int64

=== 잔차 큰 샘플 Duration 값 ===
      Exercise_Duration    BPM  Weight(lb)  Body_Temperature(F)  \
6889               30.0  101.0       176.4                106.2   
742                29.0   99.0       196.2                105.8   
7405               25.0  118.0       207.2                105.4   
5011                2.0   82.0       196.2                102.2   
4156               29.0  102.0       149.9                105.4   
6246               29.0  102.0       198.4                105.3   
1207                1.0   79.0       205.0                 99.7   
1003               28.0  101.0       187.4                106.0   
4374                1.0   70.0       191.8                 99.9 

In [52]:
male_df2 = train[male_mask2].copy()
male_df2['Temp_diff'] = male_df2['Body_Temperature(F)'] - 98.6
male_df2['Height_Total'] = male_df2['Height(Feet)'] * 12 + male_df2['Height(Remainder_Inches)']

poly5 = PolynomialFeatures(degree=5, include_bias=False)
X_male_p = poly5.fit_transform(male_df2[core_cols].values)
X_male_b = np.column_stack([X_male_p, np.ones(len(X_male_p))])
c5, _, _, _ = lstsq(X_male_b, male_df2['Calories_Burned'].values, rcond=None)
male_df2['pred'] = X_male_b @ c5
male_df2['res'] = male_df2['Calories_Burned'] - male_df2['pred']

print("Weight 구간별 잔차:")
male_df2['w_bin'] = pd.cut(male_df2['Weight(lb)'], bins=6)
print(male_df2.groupby('w_bin')['res'].agg(['mean','std','count']))

print("\nDuration x Weight 구간별 잔차 std (히트맵):")
male_df2['dur_bin'] = pd.cut(male_df2['Exercise_Duration'], bins=[0,5,10,15,20,25,30,35])
pivot = male_df2.pivot_table(values='res', index='dur_bin', columns='w_bin', aggfunc='std')
print(pivot.to_string())

Weight 구간별 잔차:
                        mean       std  count
w_bin                                        
(118.828, 147.667] -0.022077  0.221054     74
(147.667, 176.333]  0.007259  0.268447    845
(176.333, 205.0]   -0.004309  0.278010   1858
(205.0, 233.667]    0.004866  0.282975    813
(233.667, 262.333]  0.000746  0.260457    133
(262.333, 291.0]   -0.068449  0.175462      8

Duration x Weight 구간별 잔차 std (히트맵):
w_bin     (118.828, 147.667]  (147.667, 176.333]  (176.333, 205.0]  (205.0, 233.667]  (233.667, 262.333]  (262.333, 291.0]
dur_bin                                                                                                                   
(0, 5]              0.258831            0.255825          0.273824          0.279162            0.225937          0.169895
(5, 10]             0.260428            0.273241          0.281778          0.268698            0.241981               NaN
(10, 15]            0.121378            0.271688          0.283460          0.285023    

/var/folders/zd/8r845kcj6nn1ym5f96t9hpw00000gn/T/ipykernel_37709/782518147.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(male_df2.groupby('w_bin')['res'].agg(['mean','std','count']))
/var/folders/zd/8r845kcj6nn1ym5f96t9hpw00000gn/T/ipykernel_37709/782518147.py:18: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = male_df2.pivot_table(values='res', index='dur_bin', columns='w_bin', aggfunc='std')
